In [ ]:
#!/usr/bin/env python3
"""
BULK SEQUENTIAL
"""
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

from libs.cdft_1d.augmented_lda import CDFT_MODEL as CDFT
from libs.cdft_1d.external_potentials import LJ93
from libs.ml.surrogates import setDNN, setWDA, setDNNRep, load_ml_state_dicts
from libs.ml.loss import LossL1
from libs.utils import resolve_training_device, sol_dtype, tensors_to_cpu_for_storage



def update_model(x_, rho_, T_, mesh_=None, sol_=None, ml_state_dicts_=None):
    """Build CDFT model for given mesh and temperature.
    If mesh_ and sol_ are provided (e.g. mesh_bulk, sol_bulk), use them for efficient bulk (0D) computations.
    """
    if mesh_ is not None and sol_ is not None:
        mesh_local = mesh_
        sol_local = sol_
    else:
        mesh_local = {
            "BS": 1,
            "L": x_.max().item(),
            "Nx": len(x_),
            "x_bc": [x_.min().item(), x_.max().item()],
            "x": x_,
            "x_wall": x_.min() - 0.001,
            "dx": (x_[1] - x_[0]).to(device),
        }
        sol_local = sol
    eq_params_local = dict(eq_params)
    td = sol_dtype(sol_local)
    eq_params_local["beta"] = torch.tensor(1 / T_, dtype=td, device=device)[None, None]
    eq_params_local["mu"] = rho_.sum(-1) * mesh_local["dx"]
    eq_params_local["Vext"] = LJ93(mesh_local["x"], mesh_local["x_wall"], 1, 2)[None, ...].to(device) * 0
    model = CDFT(eq_params_local, mesh_local, sol_local)
    sd = ml_state_dicts_ if ml_state_dicts_ is not None else None
    setDNN(model, LR=0.0, state_dicts=sd)
    setDNNRep(model, LR=0.0, state_dict=sd["dnn_rep_fn"] if sd else None)
    setWDA(model, LR=0.0, modes=150, state_dict=sd["wda_fn"] if sd else None)
    return model


"""def chem_pot(rho, model):
    beta = model.eq_params["beta"][..., None]
    Lambda = model.eq_params["Lambda"]
    DF = model.gradients_FX_bulk(rho, detach_tensors=True, compute_D2FX=False)["DF"]
    mu = 1 / beta * (torch.log(Lambda**3 * rho)) + DF
    return mu
"""

def coex_equations_auto(rho_vec, model):
    """Returns [eq1, eq2] enforcing mu(rho1)=mu(rho2) and P(rho1)=P(rho2)."""
    r1, r2 = rho_vec
    R1 = r1 * torch.ones_like(model.sol["rho_guess"])
    R2 = r2 * torch.ones_like(model.sol["rho_guess"])
    mu1 = model.GetChemPot_bulk(R1).squeeze()
    mu2 = model.GetChemPot_bulk(R2).squeeze()
    p1 = -model.Get_omega_bulk(R1)[0, 0]
    p2 = -model.Get_omega_bulk(R2)[0, 0]
    return [mu1 - mu2, p1 - p2]


def solve_newton(coex_equations, rho_init, model, tol=1e-5, max_iter=1000, alpha=0.3, verbose=True):
    """Newton-Raphson for coexistence equations."""
    rho0 = rho_init.clone().detach().to(model.sol["device"])
    for i in range(max_iter):
        rho0.requires_grad_(True)
        rho = rho0
        F = coex_equations(rho, model)
        td = sol_dtype(model.sol)
        J = torch.zeros((2, 2), dtype=td, device=rho.device)
        for j in range(2):
            J_i = torch.autograd.grad(
                F[j], rho, retain_graph=True, materialize_grads=True
            )[0]
            J[j, :] = J_i
        F_n = torch.tensor([F[0].item(), F[1].item()], dtype=td, device=rho.device)
        with torch.no_grad():
            delta_rho = torch.linalg.solve(J, -F_n)
            rho0.add_(alpha * delta_rho)
        if verbose and i % 10 == 0:
            print(i, delta_rho.norm().item(), rho0)
        if torch.norm(delta_rho).item() < tol:
            if verbose:
                print("Number of Newton iterations:", i + 1)
            break
    return rho0.detach()


def newton_critical_point(
    eq_params, mesh, sol, rho0, T0, max_iter=20, tol=1e-8, alpha=1., device=None, ml_state_dicts_=None
):
    """Solve for (rho_c, T_c) such that ∂p/∂rho=0 and ∂²p/∂rho²=0."""
    if device is None:
        device = sol["device"]
    device = torch.device(device)
    td = sol_dtype(sol)
    rho = torch.tensor(float(rho0), dtype=td, device=device)
    T = torch.tensor(float(T0), dtype=td, device=device)
    # Vext matching mesh (zero for bulk)
    Vext_mesh = eq_params["Vext"]
    for it in range(max_iter):
        rho_var = rho.clone().detach().requires_grad_(True)
        T_var = T.clone().detach().requires_grad_(True)
        beta = 1.0 / T_var
        eq_params_local = dict(eq_params)
        eq_params_local["beta"] = beta * torch.ones_like(eq_params_local["mu"], dtype=td)
        eq_params_local["Vext"] = Vext_mesh
        model = CDFT(eq_params_local, mesh, sol)
        sd = ml_state_dicts_
        setDNN(model, LR=0.0, state_dicts=sd)
        setDNNRep(model, LR=0.0, state_dict=sd["dnn_rep_fn"] if sd else None)
        setWDA(model, LR=0.0, modes=150, state_dict=sd["wda_fn"] if sd else None)
        model.dnn_fn.eval()
        model.dnn_rep_fn.eval()
        model.wda_fn.eval()
        R1 = rho_var * torch.ones_like(model.sol["rho_guess"])
        
        omega = model.Get_omega_bulk(R1)
        p = -omega[0, 0]
        p_prime = torch.autograd.grad(p, rho_var, create_graph=True)[0]
        p_second = torch.autograd.grad(p_prime, rho_var, create_graph=True)[0]
        f1, f2 = p_prime, p_second
        F = torch.stack([f1, f2])
        if F.norm().item() < tol:
            print(f"Converged in {it} iterations (function norm).")
            break
        df1_drho = torch.autograd.grad(f1, rho_var, retain_graph=True)[0]
        df1_dT = torch.autograd.grad(f1, T_var, retain_graph=True)[0]
        df2_drho = torch.autograd.grad(f2, rho_var, retain_graph=True)[0]
        df2_dT = torch.autograd.grad(f2, T_var)[0]
        J = torch.stack([torch.stack([df1_drho, df1_dT]), torch.stack([df2_drho, df2_dT])])
        delta = torch.linalg.solve(J, -F)
        rho = rho + delta[0]
        T = T + delta[1]
        print("Norm:", delta.norm().item())
        if delta.norm().item() < tol:
            print(f"Converged in {it + 1} iterations (step size).")
            break
    return rho.item(), T.item()


# ------------------------------------------------------------------------------
# Main
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    plt.rcParams["text.usetex"] = False

    # ------------------------------------------------------------------------------
    # Output directory
    # ------------------------------------------------------------------------------

    script_dir = os.path.dirname(os.path.abspath("./src"))
    sys.path.insert(0, script_dir)
    output_dir = os.path.join(script_dir, "..", "output", "bulk_tmd")
    os.makedirs(output_dir, exist_ok=True)

    # ------------------------------------------------------------------------------
    # Configuration
    # ------------------------------------------------------------------------------
    outdir = os.path.join(script_dir, "..", "output") + "/"
    datadir = os.path.join(script_dir, "..", "data") + "/"
    pkldir = datadir + "dataset/pkl/profiles/"
    DEVICE_KIND = "auto"  # or "cuda" | "mps" | "cpu"; override with env TORCH_DEVICE
    device = resolve_training_device(DEVICE_KIND)
    print(f"Using device: {device}")
    TORCH_DTYPE = torch.float64
    torch.manual_seed(42)

    # Flags
    JACOBIAN = "EXACT"
    USE_MODEL = 1
    USE_DBH_DIAMETER = 1
    TRAIN_DNN = 0
    TRAIN_WDA = 0
    RESTART_ML_MODEL = 1
    SAVE_MODEL = 1
    fast = False

    # Thermodynamic parameters
    R, mu, m = 1., 1., 1.
    beta = 1 / 1.6
    dft_type = "LDA"
    ensemble = "NVT"
    guess_coex = [0.001, 0.9]
    str_param = "mu" if ensemble == "muVT" else "N"
    Lambda = 1.

    # External potential parameters
    sigma_attr = R
    eps_attr = 1.
    cutoff_attr = 2.5 * R
    Ew = 1.
    sigmaw = 1 * sigma_attr

    # Mesh (full-size, kept for compatibility with model init)
    L = 30.
    xmin, xmax = -L, L
    Nx = int(2 * L / (0.2 * R))
    x = torch.linspace(xmin, xmax, Nx, dtype=TORCH_DTYPE).to(device)
    dx = x[1] - x[0]
    x_wall = xmin - 0.01 * sigmaw
    BS = 1

    # External potential and initial guess
    Vext = torch.zeros((BS, 1), dtype=TORCH_DTYPE).to(device)
    rho_guess = torch.zeros((BS, 1), dtype=TORCH_DTYPE).to(device)
    rho_guess_bulk = torch.zeros((BS, 1), dtype=TORCH_DTYPE).to(device)

    # Build parameter dicts
    eq_params = {
        "R": torch.tensor(R, dtype=TORCH_DTYPE, device=device),
        "mu": torch.tensor(mu, dtype=TORCH_DTYPE, device=device)[None, None],
        "beta": torch.tensor(beta, dtype=TORCH_DTYPE, device=device)[None, None],
        "Lambda": torch.tensor(Lambda, dtype=TORCH_DTYPE, device=device),
        "dft_type": dft_type,
        "ensemble": ensemble,
        "str_param": str_param,
        "sigma_attr": torch.tensor(sigma_attr, dtype=TORCH_DTYPE, device=device),
        "eps_attr": torch.tensor(eps_attr, dtype=TORCH_DTYPE, device=device),
        "cutoff_attr": torch.tensor(cutoff_attr, dtype=TORCH_DTYPE, device=device),
        "Ew": torch.tensor(Ew, dtype=TORCH_DTYPE, device=device),
        "sigmaw": torch.tensor(sigmaw, dtype=TORCH_DTYPE, device=device),
        "Vext": Vext,
        "BC_R": "NONE",
        "fast": fast,
    }

    mesh = {
        "BS": BS,
        "L": L,
        "Nx": Nx,
        "x_bc": [xmin, xmax],
        "x": x,
        "x_wall": torch.tensor(x_wall, dtype=TORCH_DTYPE, device=device),
        "dx": dx.to(device),
    }


    sol = {
        "rho_guess": rho_guess,
        "device": device,
        "dtype": TORCH_DTYPE,
        "outdir": outdir,
        "datadir": datadir,
        "pkldir": pkldir,
        "JACOBIAN": JACOBIAN,
        "RESTART_ML_MODEL": RESTART_ML_MODEL,
        "SAVE_MODEL": SAVE_MODEL,
        "USE_MODEL": USE_MODEL,
        "USE_DBH_DIAMETER": USE_DBH_DIAMETER,
        "TRAIN_DNN": TRAIN_DNN,
        "TRAIN_WDA": TRAIN_WDA,
        "LOSS": LossL1,
    }

    # sol_bulk: same as sol but with minimal rho_guess for bulk (0D) computations
    sol_bulk = {**sol, "rho_guess": rho_guess_bulk}

    # Load ML state dicts once to avoid repeated disk I/O in coexistence/critical-point loops
    ml_state_dicts = load_ml_state_dicts(datadir, device) if RESTART_ML_MODEL else None

    model = CDFT(eq_params, mesh, sol)
    sd = ml_state_dicts
    setDNN(model, LR=0, state_dicts=sd)
    setDNNRep(model, LR=0, state_dict=sd["dnn_rep_fn"] if sd else None)
    setWDA(model, LR=0, modes=150, state_dict=sd["wda_fn"] if sd else None)

    # Critical point NN (bulk mesh with BULK_COMP for efficiency)
    sol_bulk["USE_MODEL"] = True
    rho_c_guess, T_c_guess = 0.3190, 1.0779
    rho_c_nn, T_c_nn = newton_critical_point(
        eq_params, mesh, sol_bulk, rho_c_guess, T_c_guess,
        max_iter=150, tol=1e-9, alpha=1., device=device, ml_state_dicts_=ml_state_dicts
    )
    print("Critical point (NN):")
    print("rho_c =", rho_c_nn)
    print("T_c   =", T_c_nn)

    # Critical point MF (USE_MODEL=False)
    sol_bulk["USE_MODEL"] = False
    rho_c_guess_mf, T_c_guess_mf = 0.08, 0.95  # MF/LDA critical point is at lower rho, T
    rho_c_mf, T_c_mf = newton_critical_point(
        eq_params, mesh, sol_bulk, rho_c_guess_mf, T_c_guess_mf,
        max_iter=150, tol=1e-9, alpha=1., device=device, ml_state_dicts_=None
    )
    print("Critical point (MF):")
    print("rho_c =", rho_c_mf)
    print("T_c   =", T_c_mf)

    # Coexistence without neural model (USE_MODEL=False) - use minimal bulk mesh for ~35x speedup
    # T_list0 = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]
    T_list0 = [0.55, 0.65, 0.75, 0.85,0.95]

    rho_l_l0, rho_v_l0 = [], []
    for T_val in T_list0:
        model = update_model(x_=mesh["x"], rho_=rho_guess_bulk, T_=T_val, mesh_=mesh, sol_=sol_bulk, ml_state_dicts_=ml_state_dicts)
        model.sol["USE_MODEL"] = False
        print(f"T={1/model.eq_params['beta'].item():.2f} (MF)")
        rho_v, rho_l = solve_newton(
            coex_equations_auto,
            rho_init=torch.tensor([1e-15, 1 - 1e-15], dtype=TORCH_DTYPE, device=device),
            model=model,
            tol=1e-7,
            max_iter=1000,
            alpha=0.9,
            verbose=False,
        )
        print(f"rho_v = {rho_v.item()}, rho_l = {rho_l.item()}")
        rho_v_l0.append(rho_v), rho_l_l0.append(rho_l)

        
    # Restore USE_MODEL for coexistence loops
    sol_bulk["USE_MODEL"] = True

    # Coexistence with neural model (USE_MODEL=True) - bulk mesh with BULK_COMP
    # T_list = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]
    T_list = [0.55, 0.65, 0.75, 0.85,0.95]

    rho_l_l, rho_v_l = [], []
    for T_val in T_list:
        model = update_model(x_=mesh["x"], rho_=rho_guess_bulk, T_=T_val, mesh_=mesh, sol_=sol_bulk, ml_state_dicts_=ml_state_dicts)
        model.sol["USE_MODEL"] = True
        print(f"T={1/model.eq_params['beta'].item():.2f}")
        rho_v, rho_l = solve_newton(
            coex_equations_auto,
            rho_init=torch.tensor([1e-15, 1 - 1e-15], dtype=TORCH_DTYPE, device=device),
            model=model,
            tol=1e-6,
            max_iter=100000,
            alpha=0.9,
            verbose=False,
        )
        print(f"rho_v = {rho_v.item()}, rho_l = {rho_l.item()}")
        rho_v_l.append(rho_v), rho_l_l.append(rho_l)


    # Literature MD data
    T_md = [0.64, 0.67, 0.70, 0.73, 0.76, 0.79, 0.82, 0.85, 0.88, 0.91, 0.94, 0.97, 1.00, 1.03, 1.06]
    rho_l_md = [0.8176, 0.8024, 0.7866, 0.7704, 0.7538, 0.7361, 0.7181, 0.6986, 0.6784, 0.6556, 0.6309, 0.6032, 0.5712, 0.530, 0.463]
    rho_v_md = [0.00351, 0.00525, 0.00727, 0.01036, 0.01374, 0.01776, 0.0233, 0.0303, 0.0392, 0.0483, 0.0616, 0.0763, 0.096, 0.127, 0.168]
    T_md.append(1.085)
    rho_v_md.append(0.3170)
    rho_l_md.append(0.3170)

    # Build DataFrames
    pc_nn = pd.DataFrame(columns=["T", "rho_l", "rho_v"]).set_index(["T"])
    pc_0 = pd.DataFrame(columns=["T", "rho_l", "rho_v"]).set_index(["T"])
    pc_md = pd.DataFrame(columns=["T", "rho_l", "rho_v"]).set_index(["T"])

    for i, T_val in enumerate(T_list):
        pc_nn.at[(T_val), "rho_v"] = rho_v_l[i].item()
        pc_nn.at[(T_val), "rho_l"] = rho_l_l[i].item()
    for i, T_val in enumerate(T_list0):
        pc_0.at[(T_val), "rho_v"] = rho_v_l0[i].item()
        pc_0.at[(T_val), "rho_l"] = rho_l_l0[i].item()
    for i, T_val in enumerate(T_md):
        pc_md.at[(T_val), "rho_v"] = rho_v_md[i]
        pc_md.at[(T_val), "rho_l"] = rho_l_md[i]

    # Convert tensors to floats for plotting (ensures correct MF bulk display)
    rho_v_l0_plt = [r.item() if torch.is_tensor(r) else r for r in rho_v_l0]
    rho_l_l0_plt = [r.item() if torch.is_tensor(r) else r for r in rho_l_l0]
    rho_v_l_plt = [r.item() if torch.is_tensor(r) else r for r in rho_v_l]
    rho_l_l_plt = [r.item() if torch.is_tensor(r) else r for r in rho_l_l]

    # Plot: solid curves (excl. last 2 pts for MF to avoid overlap with dotted), dotted to critical
    plt.figure(figsize=(8, 6))
    plt.plot(rho_v_l0_plt[:-1], T_list0[:-1], color="blue")
    plt.plot(rho_l_l0_plt[:-1], T_list0[:-1], color="blue", label="rho - MF")
    plt.plot(rho_v_l_plt, T_list, color="red")
    plt.plot(rho_l_l_plt, T_list, color="red", label="rho - NN")
    plt.plot(rho_v_md[:-1], T_md[:-1], "--", color="black")
    plt.plot(rho_l_md[:-1], T_md[:-1], "--", color="black", label="rho - MD")
    # Dotted: connect coexistence curve to critical point (NN and MF)
    # NN: last point to critical point
    plt.plot([rho_v_l_plt[-1], rho_c_nn], [T_list[-1], T_c_nn], ":", color="red", linewidth=2, zorder=5)
    plt.plot([rho_l_l_plt[-1], rho_c_nn], [T_list[-1], T_c_nn], ":", color="red", linewidth=2, zorder=5)
    # MF: last 2 points + critical point (longer dotted segment)
    plt.plot([rho_v_l0_plt[-2], rho_v_l0_plt[-1], rho_c_mf], [T_list0[-2], T_list0[-1], T_c_mf], ":", color="blue", linewidth=2, zorder=5)
    plt.plot([rho_l_l0_plt[-2], rho_l_l0_plt[-1], rho_c_mf], [T_list0[-2], T_list0[-1], T_c_mf], ":", color="blue", linewidth=2, zorder=5)
    # MD: last point is the critical point
    plt.plot(rho_v_md[-2:], T_md[-2:], ":", color="black")
    plt.plot(rho_l_md[-2:], T_md[-2:], ":", color="black")
    plt.scatter(rho_c_nn, T_c_nn, c="red", label="T_c - NN")
    plt.scatter(rho_v_md[-1], T_md[-1], c="black", label="T_c - MD")
    plt.scatter(rho_c_mf, T_c_mf, c="blue", label="T_c - MF")
    plt.ylabel("T")
    plt.xlabel(r"$\rho$")
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(output_dir, "phase_curve_try.svg"))
    #plt.savefig(os.path.join(output_dir, "phase_curve_try.png"))
    plt.close()

    # Save pickle files
    tensors_to_cpu_for_storage(pc_nn).to_pickle(os.path.join(output_dir, "pc_nn_operative.pkl"))
    tensors_to_cpu_for_storage(pc_md).to_pickle(os.path.join(output_dir, "pc_md_operative.pkl"))
    tensors_to_cpu_for_storage(pc_0).to_pickle(os.path.join(output_dir, "pc_0_operative.pkl"))

    # Save critical points
    with open(os.path.join(output_dir, "critical_point.txt"), "w") as f:
        f.write(f"NN: rho_c = {rho_c_nn}\nNN: T_c   = {T_c_nn}\n")
        f.write(f"MF: rho_c = {rho_c_mf}\nMF: T_c   = {T_c_mf}\n")

    print(f"\nOutputs saved to {os.path.abspath(output_dir)}")
    print("\npc_nn (neural model):")
    print(pc_nn)


Using device: cuda
Norm: 0.08019121941569236
Norm: 0.031248701411200865
Norm: 0.0008573750992671306
Norm: 5.176526039665658e-07
Converged in 4 iterations (function norm).
Critical point (NN):
rho_c = 0.26160110781284174
T_c   = 1.0772037902003397
Norm: 0.44990190030807514
Norm: 0.35813281309353023
Norm: 0.0032770286105573473
Norm: 2.501159486320846e-05
Converged in 4 iterations (function norm).
Critical point (MF):
rho_c = 0.26650963144043593
T_c   = 1.0041109399112882
T=0.55 (MF)
rho_v = 0.003722365331036661, rho_l = 0.8394093064554266
T=0.65 (MF)
rho_v = 0.012711217798360458, rho_l = 0.7520955048090275
T=0.75 (MF)
rho_v = 0.03144995252805893, rho_l = 0.6611248154144066
T=0.85 (MF)
rho_v = 0.06605045390506491, rho_l = 0.55990670459809
T=0.95 (MF)
rho_v = 0.1347168531013158, rho_l = 0.4296897918313779
T=0.55
rho_v = 0.002552683713157041, rho_l = 0.8622704087050955
T=0.65
rho_v = 0.007256014367006196, rho_l = 0.8121692290474455
T=0.75
rho_v = 0.017036000602615436, rho_l = 0.758143567842

In [ ]:
#!/usr/bin/env python3
"""
NON-BULK SEQUENTIAL
"""
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

from libs.cdft_1d.augmented_lda import CDFT_MODEL as CDFT
from libs.cdft_1d.external_potentials import LJ93
from libs.ml.surrogates import setDNN, setWDA, setDNNRep, load_ml_state_dicts
from libs.ml.loss import LossL1
from libs.utils import resolve_training_device, sol_dtype, tensors_to_cpu_for_storage


# ------------------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------------------
def _as_batched_temperature(T_, dtype, device):
    """
    Convert temperature input to shape (B, 1).
    Accepts scalar, list, np.ndarray, or tensor.
    """
    T_ = torch.as_tensor(T_, dtype=dtype, device=device)
    if T_.ndim == 0:
        T_ = T_[None]
    return T_.reshape(-1, 1)


def _expand_rho_guess(rho_guess, batch_size):
    """
    Ensure rho_guess has shape (B, 1, Nx).
    """
    if rho_guess.shape[0] == batch_size:
        return rho_guess
    if rho_guess.shape[0] == 1:
        return rho_guess.expand(batch_size, -1, -1).clone()
    raise ValueError(
        f"rho_guess batch mismatch: got {rho_guess.shape[0]}, expected 1 or {batch_size}"
    )


def _make_zero_vext(mesh_local, batch_size, dtype, device):
    """
    Build zero external potential with shape (B, 1, Nx).
    """
    # keep same construction path as original code
    vext = LJ93(mesh_local["x"], mesh_local["x_wall"], 1, 2)[None, ...].to(device=device, dtype=dtype) * 0
    if vext.ndim == 2:
        vext = vext[:, None, :]
    return vext.expand(batch_size, -1, -1).clone()


def _clone_mesh_with_batch(mesh_local, batch_size):
    mesh_b = dict(mesh_local)
    mesh_b["BS"] = batch_size
    return mesh_b


def _clone_sol_with_rho_guess(sol_local, rho_guess_batched):
    sol_b = dict(sol_local)
    sol_b["rho_guess"] = rho_guess_batched
    return sol_b


# ------------------------------------------------------------------------------
# Model builders
# ------------------------------------------------------------------------------
def update_model(x_, rho_, T_, mesh_=None, sol_=None, ml_state_dicts_=None):
    """
    Build CDFT model for given mesh and temperature(s).

    Supports:
        rho_: (B, 1, Nx)
        T_:   scalar or (B,)
    """
    if mesh_ is not None and sol_ is not None:
        mesh_local = mesh_
        sol_local = sol_
    else:
        batch_size = rho_.shape[0]
        mesh_local = {
            "BS": batch_size,
            "L": x_.max().item(),
            "Nx": len(x_),
            "x_bc": [x_.min().item(), x_.max().item()],
            "x": x_,
            "x_wall": x_.min() - 0.001,
            "dx": (x_[1] - x_[0]).to(device),
        }
        sol_local = sol

    td = sol_dtype(sol_local)
    dev = sol_local["device"]

    if rho_.ndim != 3:
        raise ValueError(f"rho_ must have shape (B, 1, Nx), got {tuple(rho_.shape)}")

    batch_size = rho_.shape[0]
    T_b = _as_batched_temperature(T_, dtype=td, device=dev)  # (B, 1)
    if T_b.shape[0] != batch_size:
        raise ValueError(
            f"Temperature batch mismatch: T has batch {T_b.shape[0]} but rho has batch {batch_size}"
        )

    mesh_b = _clone_mesh_with_batch(mesh_local, batch_size)
    sol_b = _clone_sol_with_rho_guess(sol_local, rho_)

    eq_params_local = dict(eq_params)
    eq_params_local["beta"] = 1.0 / T_b                               # (B, 1)
    eq_params_local["mu"] = rho_.sum(dim=-1) * mesh_b["dx"]          # (B, 1)
    eq_params_local["Vext"] = _make_zero_vext(mesh_b, batch_size, td, dev)

    model = CDFT(eq_params_local, mesh_b, sol_b)

    sd = ml_state_dicts_ if ml_state_dicts_ is not None else None
    setDNN(model, LR=0.0, state_dicts=sd)
    setDNNRep(model, LR=0.0, state_dict=sd["dnn_rep_fn"] if sd else None)
    setWDA(model, LR=0.0, modes=150, state_dict=sd["wda_fn"] if sd else None)

    if hasattr(model, "dnn_fn") and model.dnn_fn is not None:
        model.dnn_fn.eval()
    if hasattr(model, "dnn_rep_fn") and model.dnn_rep_fn is not None:
        model.dnn_rep_fn.eval()
    if hasattr(model, "wda_fn") and model.wda_fn is not None:
        model.wda_fn.eval()

    return model


def chem_pot(rho, model):
    """
    Bulk chemical potential.

    rho: (B, 1, Nx)
    returns: same broadcasted shape as DF / rho
    """
    beta = model.eq_params["beta"][..., None]  # (B, 1, 1)
    Lambda = model.eq_params["Lambda"]
    DF = model.gradients_FX(rho, detach_tensors=True, compute_D2FX=False)["DF"]
    mu = (1.0 / beta) * torch.log(Lambda**3 * rho) + DF
    return mu


# ------------------------------------------------------------------------------
# Batched coexistence system
# ------------------------------------------------------------------------------
def coex_equations_auto(rho_vec, model):
    """
    Batched coexistence equations.

    Parameters
    ----------
    rho_vec : tensor, shape (B, 2)
        [:, 0] = vapor guess
        [:, 1] = liquid guess

    Returns
    -------
    F : tensor, shape (B, 2)
        F[:, 0] = mu(rho_v) - mu(rho_l)
        F[:, 1] = p(rho_v)  - p(rho_l)
    """
    if rho_vec.ndim != 2 or rho_vec.shape[1] != 2:
        raise ValueError(f"rho_vec must have shape (B, 2), got {tuple(rho_vec.shape)}")

    rho_template = model.sol["rho_guess"]              # (B, 1, Nx)
    dx = model.mesh["dx"]
    mid = rho_template.shape[-1] // 2

    r_v = rho_vec[:, 0].view(-1, 1, 1)                 # (B, 1, 1)
    r_l = rho_vec[:, 1].view(-1, 1, 1)                 # (B, 1, 1)

    R_v = r_v * torch.ones_like(rho_template)          # (B, 1, Nx)
    R_l = r_l * torch.ones_like(rho_template)          # (B, 1, Nx)

    N_v = R_v.sum(dim=-1) * dx                         # (B, 1)
    N_l = R_l.sum(dim=-1) * dx                         # (B, 1)

    mu_v = model.GetChemPot(N_v, R_v).reshape(-1)      # (B,)
    mu_l = model.GetChemPot(N_l, R_l).reshape(-1)      # (B,)

    omega_v = model.GetOmega(R_v)[0]                   # expected (B, 1, Nx)
    omega_l = model.GetOmega(R_l)[0]

    p_v = -omega_v[:, 0, mid]                          # (B,)
    p_l = -omega_l[:, 0, mid]                          # (B,)

    return torch.stack([mu_v - mu_l, p_v - p_l], dim=-1)  # (B, 2)


def solve_newton_batched(
    coex_equations,
    rho_init,
    model,
    tol=1e-5,
    max_iter=1000,
    alpha=0.3,
    clamp_min=1e-15,
    clamp_max=1.0 - 1e-15,
    verbose=True,
):
    """
    Batched Newton-Raphson for coexistence equations.

    Parameters
    ----------
    rho_init : tensor, shape (B, 2)

    Returns
    -------
    rho : tensor, shape (B, 2)
    """
    td = sol_dtype(model.sol)
    dev = model.sol["device"]

    rho0 = rho_init.clone().detach().to(device=dev, dtype=td)
    if rho0.ndim != 2 or rho0.shape[1] != 2:
        raise ValueError(f"rho_init must have shape (B, 2), got {tuple(rho0.shape)}")

    batch_size = rho0.shape[0]

    for i in range(max_iter):
        rho = rho0.detach().clone().requires_grad_(True)    # (B, 2)
        F = coex_equations(rho, model)                      # (B, 2)

        J = torch.zeros((batch_size, 2, 2), dtype=td, device=dev)

        # Because samples are independent across batch, grad(sum(F[:, j]), rho)
        # gives per-sample gradients stacked in shape (B, 2).
        for j in range(2):
            grad_j = torch.autograd.grad(
                F[:, j].sum(),
                rho,
                retain_graph=True,
                materialize_grads=True,
            )[0]                                            # (B, 2)
            J[:, j, :] = grad_j

        # Solve J * delta = -F independently for each batch item.
        delta_rho = torch.linalg.solve(J, -F.unsqueeze(-1)).squeeze(-1)   # (B, 2)

        with torch.no_grad():
            rho0 = rho0 + alpha * delta_rho
            rho0[:, 0].clamp_(min=clamp_min, max=clamp_max)
            rho0[:, 1].clamp_(min=clamp_min, max=clamp_max)

        step_norm = torch.linalg.norm(delta_rho, dim=-1)    # (B,)
        max_step = step_norm.max().item()

        if verbose and (i % 10 == 0 or max_step < tol):
            print(
                f"iter={i:4d} | max||delta||={max_step:.3e} | "
                f"mean||delta||={step_norm.mean().item():.3e}"
            )

        if max_step < tol:
            if verbose:
                print(f"Converged in {i + 1} Newton iterations.")
            break

    return rho0.detach()


def batched_coexistence_temperatures(
    T_list,
    mesh_bulk,
    sol_bulk,
    ml_state_dicts,
    use_model=True,
    rho_init_pair=(1e-15, 1.0 - 1e-15),
    tol=1e-7,
    max_iter=100000,
    alpha=0.9,
    chunk_size=None,
    verbose=True,
):
    """
    Solve coexistence for many temperatures in batches.

    Returns
    -------
    rho_v : tensor, shape (len(T_list),)
    rho_l : tensor, shape (len(T_list),)
    """
    td = sol_dtype(sol_bulk)
    dev = sol_bulk["device"]

    T_all = torch.as_tensor(T_list, dtype=td, device=dev)
    nT = T_all.numel()

    if chunk_size is None:
        chunk_size = nT

    rho_v_chunks = []
    rho_l_chunks = []

    for start in range(0, nT, chunk_size):
        stop = min(start + chunk_size, nT)
        T_chunk = T_all[start:stop]
        B = T_chunk.shape[0]

        rho_guess_b = _expand_rho_guess(sol_bulk["rho_guess"], B)
        sol_chunk = _clone_sol_with_rho_guess(sol_bulk, rho_guess_b)
        model = update_model(
            x_=mesh_bulk["x"],
            rho_=rho_guess_b,
            T_=T_chunk,
            mesh_=mesh_bulk,
            sol_=sol_chunk,
            ml_state_dicts_=ml_state_dicts,
        )
        model.sol["USE_MODEL"] = bool(use_model)

        rho_init = torch.tensor(rho_init_pair, dtype=td, device=dev).view(1, 2).expand(B, -1).clone()

        if verbose:
            mode = "NN" if use_model else "MF"
            print(f"\nSolving coexistence batch {start}:{stop} ({mode})")
            print("Temperatures:", [float(t) for t in T_chunk])

        rho_sol = solve_newton_batched(
            coex_equations=coex_equations_auto,
            rho_init=rho_init,
            model=model,
            tol=tol,
            max_iter=max_iter,
            alpha=alpha,
            verbose=verbose,
        )  # (B, 2)

        rho_v_chunks.append(rho_sol[:, 0])
        rho_l_chunks.append(rho_sol[:, 1])

        if verbose:
            for T_val, rv, rl in zip(T_chunk.tolist(), rho_sol[:, 0].tolist(), rho_sol[:, 1].tolist()):
                print(f"T={T_val:.3f} -> rho_v={rv:.8f}, rho_l={rl:.8f}")

    rho_v = torch.cat(rho_v_chunks, dim=0)
    rho_l = torch.cat(rho_l_chunks, dim=0)
    return rho_v, rho_l


# ------------------------------------------------------------------------------
# Critical point (kept scalar)
# ------------------------------------------------------------------------------
def newton_critical_point(
    eq_params, mesh, sol, rho0, T0, max_iter=20, tol=1e-8, alpha=1.0, device=None, ml_state_dicts_=None
):
    """Solve for (rho_c, T_c) such that ∂p/∂rho=0 and ∂²p/∂rho²=0."""
    if device is None:
        device = sol["device"]
    device = torch.device(device)
    td = sol_dtype(sol)

    rho = torch.tensor(float(rho0), dtype=td, device=device)
    T = torch.tensor(float(T0), dtype=td, device=device)

    Vext_mesh = LJ93(mesh["x"], mesh["x_wall"], 1, 2)[None, ...].to(device) * 0

    for it in range(max_iter):
        rho_var = rho.clone().detach().requires_grad_(True)
        T_var = T.clone().detach().requires_grad_(True)

        beta = 1.0 / T_var
        eq_params_local = dict(eq_params)
        eq_params_local["beta"] = beta * torch.ones_like(eq_params_local["mu"], dtype=td)
        eq_params_local["Vext"] = Vext_mesh

        model = CDFT(eq_params_local, mesh, sol)
        sd = ml_state_dicts_
        setDNN(model, LR=0.0, state_dicts=sd)
        setDNNRep(model, LR=0.0, state_dict=sd["dnn_rep_fn"] if sd else None)
        setWDA(model, LR=0.0, modes=150, state_dict=sd["wda_fn"] if sd else None)

        if hasattr(model, "dnn_fn") and model.dnn_fn is not None:
            model.dnn_fn.eval()
        if hasattr(model, "dnn_rep_fn") and model.dnn_rep_fn is not None:
            model.dnn_rep_fn.eval()
        if hasattr(model, "wda_fn") and model.wda_fn is not None:
            model.wda_fn.eval()

        R1 = rho_var * torch.ones_like(model.sol["rho_guess"])
        omega = model.GetOmega(R1)[0]
        Nx = omega.shape[-1]
        p = -omega[0, 0, Nx // 2]

        p_prime = torch.autograd.grad(p, rho_var, create_graph=True)[0]
        p_second = torch.autograd.grad(p_prime, rho_var, create_graph=True)[0]

        f1, f2 = p_prime, p_second
        F = torch.stack([f1, f2])

        if F.norm().item() < tol:
            print(f"Converged in {it} iterations (function norm).")
            break

        df1_drho = torch.autograd.grad(f1, rho_var, retain_graph=True)[0]
        df1_dT = torch.autograd.grad(f1, T_var, retain_graph=True)[0]
        df2_drho = torch.autograd.grad(f2, rho_var, retain_graph=True)[0]
        df2_dT = torch.autograd.grad(f2, T_var)[0]

        J = torch.stack([
            torch.stack([df1_drho, df1_dT]),
            torch.stack([df2_drho, df2_dT]),
        ])

        delta = torch.linalg.solve(J, -F)
        rho = rho + alpha * delta[0]
        T = T + alpha * delta[1]

        print("Norm:", delta.norm().item())
        if delta.norm().item() < tol:
            print(f"Converged in {it + 1} iterations (step size).")
            break

    return rho.item(), T.item()


# ------------------------------------------------------------------------------
# Main
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    plt.rcParams["text.usetex"] = False

    script_dir = os.path.dirname(os.path.abspath("src/"))
    sys.path.insert(0, script_dir)
    output_dir = os.path.join(script_dir, "..", "output", "bulk_tmd")
    os.makedirs(output_dir, exist_ok=True)

    outdir = os.path.join(script_dir, "..", "output") + "/"
    datadir = os.path.join(script_dir, "..", "data") + "/"
    pkldir = datadir + "dataset/pkl/profiles/"

    DEVICE_KIND = "auto"
    device = resolve_training_device(DEVICE_KIND)
    print(f"Using device: {device}")

    TORCH_DTYPE = torch.float64
    torch.manual_seed(42)

    # Flags
    JACOBIAN = "EXACT"
    USE_MODEL = 1
    USE_DBH_DIAMETER = 1
    TRAIN_DNN = 0
    TRAIN_WDA = 0
    RESTART_ML_MODEL = 1
    SAVE_MODEL = 1
    fast = False

    # Thermodynamic parameters
    R, mu, m = 1.0, 1.0, 1.0
    beta = 1 / 1.6
    dft_type = "LDA"
    ensemble = "NVT"
    guess_coex = [0.001, 0.9]
    str_param = "mu" if ensemble == "muVT" else "N"
    Lambda = 1.0

    # External potential parameters
    sigma_attr = R
    eps_attr = 1.0
    cutoff_attr = 2.5 * R
    Ew = 1.0
    sigmaw = 1 * sigma_attr

    # Full mesh
    L = 30.0
    xmin, xmax = -L, L
    Nx = int(2 * L / (0.2 * R))
    x = torch.linspace(xmin, xmax, Nx, dtype=TORCH_DTYPE).to(device)
    dx = x[1] - x[0]
    x_wall = xmin - 0.01 * sigmaw
    BS = 1

    # Minimal bulk mesh
    Nx_bulk = 3
    L_bulk = 0.2 * R * (Nx_bulk - 1) / 2
    x_bulk = torch.linspace(-L_bulk, L_bulk, Nx_bulk, dtype=TORCH_DTYPE).to(device)
    dx_bulk = x_bulk[1] - x_bulk[0]
    x_wall_bulk = x_bulk.min() - 0.01 * sigmaw

    Vext = LJ93(x, x_wall, Ew, sigmaw)[None, ...].to(device) * 0.0
    rho_guess = torch.zeros((BS, 1, Nx), dtype=TORCH_DTYPE).to(device)
    rho_guess_bulk = torch.zeros((1, 1, Nx_bulk), dtype=TORCH_DTYPE).to(device)

    eq_params = {
        "R": torch.tensor(R, dtype=TORCH_DTYPE, device=device),
        "mu": torch.tensor(mu, dtype=TORCH_DTYPE, device=device)[None, None],
        "beta": torch.tensor(beta, dtype=TORCH_DTYPE, device=device)[None, None],
        "Lambda": torch.tensor(Lambda, dtype=TORCH_DTYPE, device=device),
        "dft_type": dft_type,
        "ensemble": ensemble,
        "str_param": str_param,
        "sigma_attr": torch.tensor(sigma_attr, dtype=TORCH_DTYPE, device=device),
        "eps_attr": torch.tensor(eps_attr, dtype=TORCH_DTYPE, device=device),
        "cutoff_attr": torch.tensor(cutoff_attr, dtype=TORCH_DTYPE, device=device),
        "Ew": torch.tensor(Ew, dtype=TORCH_DTYPE, device=device),
        "sigmaw": torch.tensor(sigmaw, dtype=TORCH_DTYPE, device=device),
        "Vext": Vext,
        "BC_R": "NONE",
        "fast": fast,
    }

    mesh = {
        "BS": BS,
        "L": L,
        "Nx": Nx,
        "x_bc": [xmin, xmax],
        "x": x,
        "x_wall": torch.tensor(x_wall, dtype=TORCH_DTYPE, device=device),
        "dx": dx.to(device),
    }

    mesh_bulk = {
        "BS": 1,
        "L": L_bulk,
        "Nx": Nx_bulk,
        "x_bc": [x_bulk.min().item(), x_bulk.max().item()],
        "x": x_bulk,
        "x_wall": x_wall_bulk.to(device=device, dtype=TORCH_DTYPE)
        if torch.is_tensor(x_wall_bulk)
        else torch.tensor(x_wall_bulk, dtype=TORCH_DTYPE, device=device),
        "dx": dx_bulk.to(device),
        "BULK_COMP": True,
    }

    sol = {
        "rho_guess": rho_guess,
        "device": device,
        "dtype": TORCH_DTYPE,
        "outdir": outdir,
        "datadir": datadir,
        "pkldir": pkldir,
        "JACOBIAN": JACOBIAN,
        "RESTART_ML_MODEL": RESTART_ML_MODEL,
        "SAVE_MODEL": SAVE_MODEL,
        "USE_MODEL": USE_MODEL,
        "USE_DBH_DIAMETER": USE_DBH_DIAMETER,
        "TRAIN_DNN": TRAIN_DNN,
        "TRAIN_WDA": TRAIN_WDA,
        "LOSS": LossL1,
    }

    sol_bulk = {**sol, "rho_guess": rho_guess_bulk}

    ml_state_dicts = load_ml_state_dicts(datadir, device) if RESTART_ML_MODEL else None

    model = CDFT(eq_params, mesh, sol)
    sd = ml_state_dicts
    setDNN(model, LR=0, state_dicts=sd)
    setDNNRep(model, LR=0, state_dict=sd["dnn_rep_fn"] if sd else None)
    setWDA(model, LR=0, modes=150, state_dict=sd["wda_fn"] if sd else None)

    # --------------------------------------------------------------------------
    # Critical points
    # --------------------------------------------------------------------------
    sol_bulk["USE_MODEL"] = True
    rho_c_guess, T_c_guess = 0.3190, 1.0779
    rho_c_nn, T_c_nn = newton_critical_point(
        eq_params, mesh_bulk, sol_bulk, rho_c_guess, T_c_guess,
        max_iter=150, tol=1e-9, alpha=1.0, device=device, ml_state_dicts_=ml_state_dicts
    )
    print("Critical point (NN):")
    print("rho_c =", rho_c_nn)
    print("T_c   =", T_c_nn)

    sol_bulk["USE_MODEL"] = False
    rho_c_guess_mf, T_c_guess_mf = 0.08, 0.95
    rho_c_mf, T_c_mf = newton_critical_point(
        eq_params, mesh_bulk, sol_bulk, rho_c_guess_mf, T_c_guess_mf,
        max_iter=150, tol=1e-9, alpha=1.0, device=device, ml_state_dicts_=None
    )
    print("Critical point (MF):")
    print("rho_c =", rho_c_mf)
    print("T_c   =", T_c_mf)

    # --------------------------------------------------------------------------
    # Batched coexistence
    # --------------------------------------------------------------------------
    T_list = [0.55, 0.65, 0.75, 0.85,]
    T_list0 = [0.55, 0.65, 0.75, 0.85, 0.95]

    # Mean-field model
    sol_bulk["USE_MODEL"] = False
    rho_v_b0, rho_l_b0 = batched_coexistence_temperatures(
        T_list=T_list0,
        mesh_bulk=mesh_bulk,
        sol_bulk=sol_bulk,
        ml_state_dicts=ml_state_dicts,
        use_model=False,
        rho_init_pair=(1e-15, 1.0 - 1e-15),
        tol=1e-6,
        max_iter=1000,
        alpha=0.9,
        chunk_size=len(T_list0),
        verbose=True,
    )

    # Neural model
    sol_bulk["USE_MODEL"] = True
    rho_v_b, rho_l_b = batched_coexistence_temperatures(
        T_list=T_list,
        mesh_bulk=mesh_bulk,
        sol_bulk=sol_bulk,
        ml_state_dicts=ml_state_dicts,
        use_model=True,
        rho_init_pair=(1e-15, 1.0 - 1e-15),
        tol=1e-7,
        max_iter=100000,
        alpha=0.9,
        chunk_size=len(T_list),   # or smaller if memory becomes an issue
        verbose=True,
    )

    # Literature MD data
    T_md = [0.64, 0.67, 0.70, 0.73, 0.76, 0.79, 0.82, 0.85, 0.88, 0.91, 0.94, 0.97, 1.00, 1.03, 1.06]
    rho_l_md = [0.8176, 0.8024, 0.7866, 0.7704, 0.7538, 0.7361, 0.7181, 0.6986, 0.6784, 0.6556, 0.6309, 0.6032, 0.5712, 0.530, 0.463]
    rho_v_md = [0.00351, 0.00525, 0.00727, 0.01036, 0.01374, 0.01776, 0.0233, 0.0303, 0.0392, 0.0483, 0.0616, 0.0763, 0.096, 0.127, 0.168]
    T_md.append(1.085)
    rho_v_md.append(0.3170)
    rho_l_md.append(0.3170)

    # --------------------------------------------------------------------------
    # DataFrames
    # --------------------------------------------------------------------------
    pc_nn = pd.DataFrame({
        "T": T_list,
        "rho_v": rho_v_b.detach().cpu().numpy(),
        "rho_l": rho_l_b.detach().cpu().numpy(),
    }).set_index("T")

    pc_0 = pd.DataFrame({
        "T": T_list0,
        "rho_v": rho_v_b0.detach().cpu().numpy(),
        "rho_l": rho_l_b0.detach().cpu().numpy(),
    }).set_index("T")

    pc_md = pd.DataFrame({
        "T": T_md,
        "rho_v": rho_v_md,
        "rho_l": rho_l_md,
    }).set_index("T")

    # For plotting
    rho_v_l_plt = rho_v_b.detach().cpu().numpy().tolist()
    rho_l_l_plt = rho_l_b.detach().cpu().numpy().tolist()
    rho_v_l0_plt = rho_v_b0.detach().cpu().numpy().tolist()
    rho_l_l0_plt = rho_l_b0.detach().cpu().numpy().tolist()

    # --------------------------------------------------------------------------
    # Plot
    # --------------------------------------------------------------------------
    plt.figure(figsize=(8, 6))
    plt.plot(rho_v_l0_plt[:-1], T_list0[:-1], color="blue")
    plt.plot(rho_l_l0_plt[:-1], T_list0[:-1], color="blue", label="rho - MF")
    plt.plot(rho_v_l_plt, T_list, color="red")
    plt.plot(rho_l_l_plt, T_list, color="red", label="rho - NN")
    plt.plot(rho_v_md[:-1], T_md[:-1], "--", color="black")
    plt.plot(rho_l_md[:-1], T_md[:-1], "--", color="black", label="rho - MD")

    plt.plot([rho_v_l_plt[-1], rho_c_nn], [T_list[-1], T_c_nn], ":", color="red", linewidth=2, zorder=5)
    plt.plot([rho_l_l_plt[-1], rho_c_nn], [T_list[-1], T_c_nn], ":", color="red", linewidth=2, zorder=5)

    plt.plot([rho_v_l0_plt[-2], rho_v_l0_plt[-1], rho_c_mf], [T_list0[-2], T_list0[-1], T_c_mf], ":", color="blue", linewidth=2, zorder=5)
    plt.plot([rho_l_l0_plt[-2], rho_l_l0_plt[-1], rho_c_mf], [T_list0[-2], T_list0[-1], T_c_mf], ":", color="blue", linewidth=2, zorder=5)

    plt.plot(rho_v_md[-2:], T_md[-2:], ":", color="black")
    plt.plot(rho_l_md[-2:], T_md[-2:], ":", color="black")

    plt.scatter(rho_c_nn, T_c_nn, c="red", label="T_c - NN")
    plt.scatter(rho_v_md[-1], T_md[-1], c="black", label="T_c - MD")
    plt.scatter(rho_c_mf, T_c_mf, c="blue", label="T_c - MF")

    plt.ylabel("T")
    plt.xlabel(r"$\rho$")
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(output_dir, "phase_curve.svg"))
    plt.savefig(os.path.join(output_dir, "phase_curve.png"))
    plt.close()

    # --------------------------------------------------------------------------
    # Save
    # --------------------------------------------------------------------------
    tensors_to_cpu_for_storage(pc_nn).to_pickle(os.path.join(output_dir, "pc_nn_operative.pkl"))
    tensors_to_cpu_for_storage(pc_md).to_pickle(os.path.join(output_dir, "pc_md_operative.pkl"))
    tensors_to_cpu_for_storage(pc_0).to_pickle(os.path.join(output_dir, "pc_0_operative.pkl"))

    with open(os.path.join(output_dir, "critical_point.txt"), "w") as f:
        f.write(f"NN: rho_c = {rho_c_nn}\nNN: T_c   = {T_c_nn}\n")
        f.write(f"MF: rho_c = {rho_c_mf}\nMF: T_c   = {T_c_mf}\n")

    print(f"\nOutputs saved to {os.path.abspath(output_dir)}")
    print("\npc_nn (neural model):")
    print(pc_nn)

Using device: cuda
Norm: 0.07960335056820249
Norm: 0.03020562541424427
Norm: 0.0008154376882736484
Norm: 4.5506724965395707e-07
Converged in 4 iterations (function norm).
Critical point (NN):
rho_c = 0.26197225236070026
T_c   = 1.079033003061234
Norm: 0.45183210849631
Norm: 0.3592016541016573
Norm: 0.003301672125902604
Norm: 2.5270864107148394e-05
Converged in 4 iterations (function norm).
Critical point (MF):
rho_c = 0.26652642847304175
T_c   = 1.0051838343706818

Solving coexistence batch 0:5 (MF)
Temperatures: [0.55, 0.65, 0.75, 0.85, 0.95]
iter=   0 | max||delta||=1.631e-01 | mean||delta||=1.347e-01
iter=  10 | max||delta||=1.056e-01 | mean||delta||=2.260e-02
iter=  20 | max||delta||=1.872e-05 | mean||delta||=3.812e-06
iter=  22 | max||delta||=1.878e-07 | mean||delta||=3.824e-08
Converged in 23 Newton iterations.
T=0.550 -> rho_v=0.00369113, rho_l=0.83994524
T=0.650 -> rho_v=0.01262225, rho_l=0.75273504
T=0.750 -> rho_v=0.03125184, rho_l=0.66190880
T=0.850 -> rho_v=0.06563840, rho_

In [5]:
rho_v_b

tensor([0.0026, 0.0072, 0.0170, 0.0350], device='cuda:0', dtype=torch.float64)